In [36]:
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Any

import torch
from PIL import Image
from huggingface_hub import snapshot_download
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

In [40]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [41]:
!nvidia-smi

Thu Feb  5 17:30:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             27W /   70W |    6568MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
!pip -q install bitsandbytes

In [ ]:
# =========================
# CONFIG
# =========================
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
LOCAL_MODEL_DIR = Path("./models/qwen3-vl-2b-instruct")

# папка с картинками
IMAGE_DIR = Path("./image")

BATCH_SIZE = 1
MAX_SIDE = 512

DO_SAMPLE = False
NUM_BEAMS = 1

MAX_NEW_TOKENS = 200

CPU_THREADS = 8
DTYPE = torch.float32
USE_DYNAMIC_INT8 = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# =========================
# CPU OPTS
# =========================
def configure_cpu(threads: int) -> None:
    os.environ["OMP_NUM_THREADS"] = str(threads)
    os.environ["MKL_NUM_THREADS"] = str(threads)
    try:
        torch.set_num_threads(threads)
    except Exception:
        pass


In [44]:
!pip -q install bitsandbytes

In [45]:
# =========================
# MODEL DOWNLOAD/LOAD
# =========================
def _has_weights(local_dir: Path) -> bool:
    # один файл
    if (local_dir / "model.safetensors").exists():
        return True
    if (local_dir / "pytorch_model.bin").exists():
        return True

    # индекс шардов
    if (local_dir / "model.safetensors.index.json").exists():
        return True
    if (local_dir / "pytorch_model.bin.index.json").exists():
        return True

    # сами шарды
    if any(local_dir.glob("model-*.safetensors")):
        return True
    if any(local_dir.glob("pytorch_model-*.bin")):
        return True

    return False


def ensure_model_local(model_id: str, local_dir: Path) -> Path:
    local_dir.mkdir(parents=True, exist_ok=True)

    # если веса уже есть - ничего не делаем
    if _has_weights(local_dir):
        return local_dir

    # веса отсутствуют - значит качаем/докачиваем
    snapshot_download(
        repo_id=model_id,
        local_dir=str(local_dir),
        token=HF_TOKEN,
        max_workers=8,  # можно 4-16, в Colab обычно норм
    )

    if not _has_weights(local_dir):
        # если докачка "не материализовала" веса в local_dir, форсим повтор
        snapshot_download(
            repo_id=model_id,
            local_dir=str(local_dir),
            token=HF_TOKEN,
            force_download=True,
            max_workers=8,
        )

    if not _has_weights(local_dir):
        raise RuntimeError(
            f"В {local_dir.resolve()} нет весов (model.safetensors / shards). "
            f"Скачивание не завершилось или доступ ограничен."
        )

    return local_dir



def load_model_and_processor(local_dir: Path):
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        str(local_dir),
        device_map="auto",
        torch_dtype=DTYPE,
        # load_in_4bit=True,
        local_files_only=True,
    )
    processor = AutoProcessor.from_pretrained(
        str(local_dir),
        local_files_only=True,
    )

    model.eval()

    if USE_DYNAMIC_INT8:
        model = torch.ao.quantization.quantize_dynamic(
            model, {torch.nn.Linear}, dtype=torch.qint8
        )

    return model, processor


In [46]:
# =========================
# IMAGES
# =========================
def load_image_paths(image_dir: Path) -> List[Path]:
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    if not image_dir.exists():
        raise FileNotFoundError(f"IMAGE_DIR not found: {image_dir.resolve()}")

    paths = [p for p in sorted(image_dir.rglob("*")) if p.suffix.lower() in exts]
    return paths


def resize_image(path: Path, max_side: int) -> Image.Image:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        new_w = max(1, int(w * scale))
        new_h = max(1, int(h * scale))
        img = img.resize((new_w, new_h), resample=Image.BICUBIC)
    return img


def chunk_list(xs: List[Any], n: int):
    for i in range(0, len(xs), n):
        yield xs[i : i + n]



In [47]:
# =========================
# JSON HELPERS
# =========================
def extract_json(text: str) -> str:
    s = text.strip()

    # Найти первое { или [
    start = None
    for i, ch in enumerate(s):
        if ch in "{[":
            start = i
            break
    if start is None:
        raise ValueError("Не найдено начало JSON в ответе модели.")

    sub = s[start:]

    # 1) Попытка распарсить как JSON с возможным хвостом
    decoder = json.JSONDecoder()
    try:
        obj, end = decoder.raw_decode(sub)
        return sub[:end]
    except json.JSONDecodeError:
        pass

    # 2) Fallback: выделяем/чинм по скобкам
    open_ch = sub[0]
    close_ch = "}" if open_ch == "{" else "]"
    depth = 0
    in_str = False
    esc = False

    for j, ch in enumerate(sub):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue

        if ch == '"':
            in_str = True
            continue

        if ch == open_ch:
            depth += 1
        elif ch == close_ch:
            depth -= 1
            if depth == 0:
                return sub[: j + 1]

    # Если дошли до конца и JSON не закрылся - попробуем "дозакрыть"
    repaired = sub
    if in_str:
        repaired += '"'  # закрыть строку, если оборвалась
    if depth > 0:
        repaired += close_ch * depth
    return repaired


def parse_descriptions(data: Any, n: int) -> List[str]:
    """
    Достает описания из разных возможных форматов ответа модели.
    Нормализует в список строк длиной n (берет первые n).
    """
    # data должен быть dict
    if not isinstance(data, dict):
        raise ValueError("Неверный JSON: ожидается объект верхнего уровня")

    # 1) ожидаемый формат: {"descriptions": ["..."]}
    if "descriptions" in data:
        d = data["descriptions"]

        # model sometimes returns a single string when n == 1
        if isinstance(d, str) and n == 1 and d.strip():
            return [d.strip()]

        if isinstance(d, list):
            descs = [str(x).strip() for x in d if str(x).strip()]
            if len(descs) >= n:
                return descs[:n]
            raise ValueError(f"Неверный JSON: descriptions список длиной {len(descs)}, нужно {n}")

    # 2) частый формат при одной картинке: {"description": "..."}
    if "description" in data and isinstance(data["description"], str) and n == 1:
        if data["description"].strip():
            return [data["description"].strip()]

    # 3) старый формат: {"items":[{"description":"..."}]}
    if "items" in data and isinstance(data["items"], list):
        descs = []
        for it in data["items"]:
            if isinstance(it, dict) and "description" in it:
                s = str(it["description"]).strip()
                if s:
                    descs.append(s)
        if len(descs) >= n:
            return descs[:n]
        raise ValueError(f"Неверный JSON: items содержит {len(descs)} описаний, нужно {n}")

    raise ValueError("Неверный JSON: не найдено descriptions/description/items в ожидаемом виде")


In [48]:
# =========================
# INFERENCE
# =========================
def describe_batch(model, processor, batch_paths: List[Path]) -> List[Dict[str, str]]:
    cwd = Path.cwd()

    batch_urls: List[str] = []
    for p in batch_paths:
        rp = p.resolve()
        try:
            batch_urls.append(str(rp.relative_to(cwd)))
        except ValueError:
            batch_urls.append(str(rp))

    images = [resize_image(p, MAX_SIDE) for p in batch_paths]

    content = [{"type": "image", "image": img} for img in images]
    content.append(
        {
            "type": "text",
            "text": (
                "Ты ассистент, который возвращает только JSON. Никакого текста вне JSON.\n"
                "Задача: для каждого изображения выше, по порядку, дай 1 короткое предложение на русском.\n\n"
                "Формат ответа - строго один JSON-объект:\n"
                "{\n"
                '  "descriptions": ["..."]\n'
                "}\n\n"
                "Правила (обязательно):\n"
                f"- В массиве descriptions должно быть ровно {len(batch_urls)} строк.\n"
                "- Каждая строка - одно короткое предложение, без переносов строк.\n"
                "- Не используй кавычки \", если нужно - используй ' или перефразируй.\n"
                "- Никаких URL, названий сайтов, ссылок.\n"
                "- Никаких других ключей кроме descriptions.\n"
                "- Никакого markdown, никаких комментариев, никаких префиксов/суффиксов.\n\n"
                "Пример правильного ответа при одной картинке:\n"
                '{ "descriptions": ["Белая кошка сидит на подоконнике."] }\n'
            ),
        }
    )



    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=NUM_BEAMS,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    out_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    json_str = extract_json(out_text)
    data = json.loads(json_str)

    descs = parse_descriptions(data, len(batch_urls))

    items = [{"url": u, "description": d} for u, d in zip(batch_urls, descs)]
    return items


In [ ]:
def main():
    configure_cpu(CPU_THREADS)

    local_dir = ensure_model_local(MODEL_ID, LOCAL_MODEL_DIR)
    model, processor = load_model_and_processor(local_dir)

    image_paths = load_image_paths(IMAGE_DIR)
    if not image_paths:
        raise RuntimeError(f"В папке {IMAGE_DIR.resolve()} не найдено изображений (.jpg/.png/.webp).")

    out_path = Path("descriptions.jsonl")

    with out_path.open("w", encoding="utf-8") as f:
        for batch in chunk_list(image_paths, BATCH_SIZE):
            items = describe_batch(model, processor, batch)

            for it in items:
                f.write(json.dumps(it, ensure_ascii=False) + "\n")

            # чтобы память быстрее освобождалась между батчами
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"Готово. Результат записан в {out_path}")

if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]